# Kaggle Submission Notebook

This notebook contains only the code required to reproduce the competition submission CSV:
1. Final retrieval process
2. Retrieval of relevant documents for test queries
3. Optional hybrid embedding reranking over TF-IDF + BM25 candidates
4. CSV generation in Kaggle format


In [18]:
from pathlib import Path
import csv
import json
import re

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = Path('/kaggle/input/retrieval-engine-competition')
if not DATA_DIR.exists():
    for candidate in [Path('../data'), Path('data'), Path('../../data')]:
        if (candidate / 'docs.json').exists():
            DATA_DIR = candidate
            break
    else:
        DATA_DIR = Path('../data')  # local fallback

output_dir = Path.cwd()
if output_dir.name != 'kaggle' and (Path('kaggle').exists()):
    output_dir = Path('kaggle')

OUTPUT_PATH = output_dir / 'solutions_SeaFour.csv'
TOP_K = 100

# Retrieval mode:
# - 'bm25': lexical ranking with BM25+
# - 'tfidf': lexical ranking with TF-IDF cosine similarity
# - 'embedding_hybrid': build candidates with TF-IDF + BM25, then rerank them with embeddings
MODEL_NAME = 'embedding_hybrid'

# Dense reranking parameters used only when MODEL_NAME == 'embedding_hybrid'.
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
EMBEDDING_BATCH_SIZE = 128
HYBRID_CANDIDATE_K = 200



In [19]:
def value_to_text(value):
    if value is None:
        return ''
    if isinstance(value, (list, tuple)):
        return ' '.join(str(v) for v in value)
    if pd.isna(value):
        return ''
    return str(value)


def create_content_column(df, columns):
    out = df.copy()
    for col in columns:
        if col not in out.columns:
            out[col] = ''

    merged = []
    for _, row in out[columns].iterrows():
        text = ' '.join(value_to_text(row[col]) for col in columns).strip().lower()
        merged.append(text)

    out['content'] = merged
    out['id'] = out['id'].astype(str)
    return out


token_pattern = re.compile(r'[a-z0-9]+')


def tokenize(text):
    txt = str(text or '').lower()
    txt = re.sub(r'[-_/]', ' ', txt)
    return token_pattern.findall(txt)


In [20]:
def run_tfidf_search(docs_df, queries_df, top_k=100):
    top_k = min(top_k, len(docs_df))

    # TF-IDF captures exact lexical overlap and short phrases between queries and documents.
    vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)
    try:
        doc_vectors = vectorizer.fit_transform(docs_df['content'])
    except ValueError as error:
        if 'After pruning, no terms remain' not in str(error):
            raise
        vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
        doc_vectors = vectorizer.fit_transform(docs_df['content'])

    query_vectors = vectorizer.transform(queries_df['content'])

    # Cosine similarity ranks documents by how much vocabulary they share with the query.
    scores = cosine_similarity(query_vectors, doc_vectors)
    doc_ids = docs_df['id'].astype(str).to_numpy()

    results = []
    for i, row_scores in enumerate(scores):
        top_idx = np.argsort(row_scores)[-top_k:][::-1]
        results.append({
            'query_id': str(queries_df.iloc[i]['id']),
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


def run_bm25_search(docs_df, queries_df, top_k=100):
    try:
        from rank_bm25 import BM25Plus
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rank-bm25'])
        from rank_bm25 import BM25Plus

    top_k = min(top_k, len(docs_df))

    # BM25 uses token frequencies and document length normalization for lexical retrieval.
    tokenized_corpus = [tokenize(text) for text in docs_df['content']]
    bm25 = BM25Plus(tokenized_corpus)
    doc_ids = docs_df['id'].astype(str).to_numpy()

    results = []
    for _, row in queries_df.iterrows():
        query_tokens = tokenize(row['content'])
        scores = bm25.get_scores(query_tokens)
        top_idx = np.argsort(scores)[-top_k:][::-1]
        results.append({
            'query_id': str(row['id']),
            'relevant_docs': doc_ids[top_idx].tolist(),
        })
    return results


def run_embedding_hybrid_search(
    docs_df,
    queries_df,
    top_k=100,
    candidate_k=200,
    batch_size=128,
    model_name='sentence-transformers/all-MiniLM-L6-v2',
):
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])
        from sentence_transformers import SentenceTransformer

    top_k = min(top_k, len(docs_df))
    candidate_k = min(max(candidate_k, top_k), len(docs_df))

    # Step 1: use both lexical models to build a higher-recall candidate set.
    tfidf_results = run_tfidf_search(docs_df, queries_df, top_k=candidate_k)
    bm25_results = run_bm25_search(docs_df, queries_df, top_k=candidate_k)

    # Step 2: encode the full corpus once, then compare each query embedding to the candidate docs.
    model = SentenceTransformer(model_name)
    doc_embeddings = model.encode(
        docs_df['content'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    query_embeddings = model.encode(
        queries_df['content'].tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    doc_ids = docs_df['id'].astype(str).to_numpy()
    doc_id_to_index = {doc_id: idx for idx, doc_id in enumerate(doc_ids)}
    query_ids = queries_df['id'].astype(str).tolist()

    results = []
    for i, query_embedding in enumerate(query_embeddings):
        # Merge TF-IDF and BM25 ranked lists while keeping each document only once.
        candidate_doc_ids = []
        seen = set()
        for ranked_doc_ids in (
            tfidf_results[i]['relevant_docs'],
            bm25_results[i]['relevant_docs'],
        ):
            for doc_id in ranked_doc_ids:
                doc_id = str(doc_id)
                if doc_id in seen:
                    continue
                candidate_doc_ids.append(doc_id)
                seen.add(doc_id)

        candidate_indices = np.array([doc_id_to_index[doc_id] for doc_id in candidate_doc_ids], dtype=int)

        # Step 3: semantic reranking prioritizes candidates that are conceptually close to the query.
        candidate_scores = doc_embeddings[candidate_indices] @ query_embedding
        rerank_count = min(top_k, len(candidate_doc_ids))
        reranked_idx = np.argsort(candidate_scores)[-rerank_count:][::-1]

        results.append({
            'query_id': query_ids[i],
            'relevant_docs': [candidate_doc_ids[idx] for idx in reranked_idx],
        })

    return results


In [21]:
def write_kaggle_submission(results, sample_csv_path, output_csv_path):
    pred_map = {
        str(item['query_id']): [str(doc_id) for doc_id in item['relevant_docs']]
        for item in results
    }

    with open(sample_csv_path, 'r', newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        fieldnames = reader.fieldnames
        rows = list(reader)

    if fieldnames is None or len(fieldnames) < 2:
        raise ValueError('Invalid sample submission format.')

    id_col = fieldnames[0]
    pred_col = fieldnames[1]
    category_col = fieldnames[2] if len(fieldnames) >= 3 else None

    with open(output_csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in rows:
            qid = str(row[id_col])
            if qid not in pred_map:
                raise ValueError(f'Missing prediction for query_id: {qid}')

            out_row = {
                id_col: qid,
                pred_col: json.dumps(pred_map[qid]),
            }

            if category_col is not None:
                out_row[category_col] = row.get(category_col, '?') or '?'

            writer.writerow(out_row)


In [22]:
docs_df = pd.read_json(DATA_DIR / 'docs.json')
test_queries_df = pd.read_json(DATA_DIR / 'queries_test.json')
sample_submission_path = DATA_DIR / 'submission.csv'

# Build the shared text field used by all retrieval models.
docs_df = create_content_column(docs_df, ['title', 'text', 'tags'])
test_queries_df = create_content_column(test_queries_df, ['title', 'text'])

if MODEL_NAME == 'bm25':
    test_results = run_bm25_search(docs_df, test_queries_df, top_k=TOP_K)
elif MODEL_NAME == 'tfidf':
    test_results = run_tfidf_search(docs_df, test_queries_df, top_k=TOP_K)
elif MODEL_NAME == 'embedding_hybrid':
    test_results = run_embedding_hybrid_search(
        docs_df,
        test_queries_df,
        top_k=TOP_K,
        candidate_k=HYBRID_CANDIDATE_K,
        batch_size=EMBEDDING_BATCH_SIZE,
        model_name=EMBEDDING_MODEL_NAME,
    )
else:
    raise ValueError('MODEL_NAME must be "bm25", "tfidf", or "embedding_hybrid".')

write_kaggle_submission(test_results, sample_submission_path, OUTPUT_PATH)
print(f'Saved: {OUTPUT_PATH.resolve()}')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1688 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Saved: /Users/sorooshaghaei/Desktop/Paris_cite_projects/retrieval_project/kaggle/solutions_SeaFour.csv


In [23]:
submission_preview = pd.read_csv(OUTPUT_PATH)
submission_preview.head()


,query_id,relevant_doc_ids,category
0,4ffe16bc-5235-418d-9bf3-22d1f2c5796e_145437,"[""c583f3cd-b1ca-4b74-ac3e-1c6771eb6a8e_131332""...",?
1,1bb2bb20-7f45-4dcf-a94a-420c454f87b8_56473,"[""edd58474-cef1-4d75-b184-99ee27def6a2_116280""...",?
2,6a9a342c-1275-4bb3-a818-8bcce53fac4f_34507,"[""43fa6e5a-6ad6-4701-ba44-68b513409ffc_17053"",...",?
3,cb216e47-add6-41fd-974a-39251e4df3aa_6777,"[""8d766d9a-99a4-427e-88a1-2ffcdf634811_32731"",...",?
4,14f1d3f5-8271-400e-9ef2-8319de25c9a1_200748,"[""4ff296e2-e448-4f41-8e3d-4b56ac994a17_14385"",...",?
